# Análise de Sentimentos - Reviews do site Olist

O objetivo deste notebook é desenvolver e avaliar modelos de análise de sentimentos utilizando o dataset *Brazilian E-Commerce Public Dataset by Olist*, com foco na classificação de avaliações de clientes em positivas ou negativas.

Neste trabalho, são comparadas abordagens tradicionais de Machine Learning com técnicas de Deep Learning. Inicialmente, são utilizadas duas técnicas clássicas de extração de características textuais: Bag of Words e TF-IDF, aplicadas em modelos como Regressão Logística, Naive Bayes e Random Forest. Em seguida, é implementado um modelo baseado em redes neurais recorrentes do tipo LSTM (Long Short-Term Memory), com o objetivo de capturar melhor o contexto e a ordem das palavras nas avaliações.

Sobre o dataset: foi utilizado o arquivo `olist_order_reviews_dataset.csv`, que contém avaliações de compras realizadas na plataforma Olist, representando um problema real de análise de sentimentos em português.

Este notebook foi desenvolvido de forma a ser acessível para iniciantes em Processamento de Linguagem Natural (NLP), ao mesmo tempo em que apresenta técnicas relevantes utilizadas na área. Para melhor compreensão dos conceitos abordados, os seguintes materiais podem ser consultados:

- Pré-processamento de texto  
- Bag of Words e TF-IDF  
- Métricas de classificação  
- Modelos de Machine Learning (Regressão Logística, Naive Bayes e Random Forest)  
- Redes neurais recorrentes (LSTM)

Ao final, é realizada uma comparação entre as diferentes abordagens, considerando desempenho, custo computacional e capacidade de generalização dos modelos.


# Pré-processamento


### Importando as bibliotecas necessárias

In [ ]:
#!pip install spacy

In [ ]:
import pandas as pd
import re
import nltk
import spacy
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
import spacy.cli
spacy.cli.download("pt_core_news_sm")

In [ ]:
import pt_core_news_sm

spc_pt = pt_core_news_sm.load()

## Análise Exploratória de Dados(EDA)

In [ ]:
data = pd.read_csv("olist_order_reviews_dataset.csv")

In [ ]:
data.head(10)

Como nosso foco é a análise de sentimentos, não precisamos das colunas `order_id`, `review_creation_date` e `review_answer_timestamp`.

In [ ]:
data.drop(['order_id', 'review_creation_date', 'review_answer_timestamp'], axis=1, inplace=True)

In [ ]:
data.info()

Vamos checar se há dados duplicados.

In [ ]:
duplicados = round(sum(data.duplicated("review_id"))/len(data)*100, 2)
print(f"Reviews com id duplicado: {duplicados}%.")

In [ ]:
data[data.duplicated("review_id", keep =  False)].sort_values(by = "review_id")

In [ ]:
data.drop_duplicates("review_id", inplace = True) # remove os duplicados

Temos pouco menos de 100000 datapoints, todos possuindo um score de review, mas nem todos possuem review escrita.

Como estamos interessados justamente no texto, vamos juntar as colunas `review_comment_title` e `review_comment_message` em uma só e tirar entradas que não possuem texto.

In [ ]:
data.fillna('', inplace = True) # para nao ter problemas com nulos na concatenacao

In [ ]:
# concatenando as duas colunas
data['review'] = data['review_comment_title'] + ' ' + data['review_comment_message']

In [ ]:
# removendo entradas sem texto
data = data[data['review'] != ' ']

In [ ]:
data.info()

In [ ]:
data.head(10)

### Review scores

In [ ]:
data['review_score'].value_counts()

Há 5 valores de score diferentes (indo de 1 - pior até 5 - melhor), porém, para facilitar nossa tarefa, vamos classificar as reviews apenas como positiva ou negativa. Se o score for menor ou igual a 3, consideraremos negativa (0) e caso contrário, positiva (1). 

In [ ]:
labels = []

for score in data['review_score']:
  if score > 3:
    labels.append(1)
  else:
    labels.append(0)

data['label'] = labels

In [ ]:
data.head(10)

In [ ]:
plt.figure(figsize=(8,6))

sns.countplot(x=data['label'])

plt.title('Distribuição das Classes de Sentimento', fontsize=14)
plt.xlabel('Sentimento', fontsize=12)
plt.ylabel('Quantidade', fontsize=12)

plt.xticks([0, 1], ['Negativo', 'Positivo'])

plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.show()


Há bem mais reviews positivas do que negativas.

### Pré-processamento do texto

In [ ]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
stopwords_pt = stopwords.words("portuguese")

In [ ]:
stopwords_pt

Palavras como 'não' e 'nem' podem ser importantes na análise de sentimentos, por isso vamos tirá-las da lista de stopwords.

In [ ]:
stopwords_pt.remove('não')
stopwords_pt.remove('nem')

In [ ]:
def limpa_texto(texto):
  '''(str) -> str
  Essa funcao recebe uma string, deixa tudo em minusculo, filtra apenas letras,
  retira stopwords, lemmatiza e retorna a string resultante.
  '''
  texto = texto.lower()

  texto = re.sub(r"[\W\d_]+", " ", texto)

  texto = [pal for pal in texto.split() if pal not in stopwords_pt]

  spc_texto = spc_pt(" ".join(texto))
  tokens = [word.lemma_ if word.lemma_ != "-PRON-" else word.lower_ for word in spc_texto]
  
  return " ".join(tokens)

In [ ]:
data['review'] = data['review'].apply(limpa_texto)

In [ ]:
data.info()

In [ ]:
data.head(10)

In [ ]:
data[data['review'] == '']

Como tinham reviews com apenas números ou símbolos, ainda há dados faltantes na coluna `review`, vamos removê-los.

In [ ]:
data = data[data['review'] != '']

In [ ]:
data.info()

In [ ]:
# rode essa celula se quiser salvar o dataset pre-processado
data.to_csv('olist_preprocessado.csv', index= False, columns= ['review_id', 'review', 'label'])

# Feature extraction

Nesta etapa, são testadas duas abordagens clássicas de representação de texto: Bag of Words e TF-IDF.

O método Bag of Words transforma os textos em vetores numéricos baseados na presença ou ausência de palavras, podendo utilizar uma representação binária (0 ou 1). Já o TF-IDF (Term Frequency–Inverse Document Frequency) leva em consideração não apenas a frequência das palavras em um documento, mas também sua importância em relação ao conjunto de textos.

A comparação entre essas duas técnicas permite avaliar como diferentes formas de representação textual impactam o desempenho dos modelos de classificação.


### Com Bag of Words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# Instanciando o CountVectorizer, binary=True faz a codificacao binaria
vectorizer = CountVectorizer(binary=True, max_features=5000)

texto = data['review']

# Vetorizando o texto
X_bow = vectorizer.fit_transform(texto)

In [ ]:
X_bow.toarray()

In [ ]:
print(X_bow.shape, type(X_bow))

### Com TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Instanciando o TfidfVectorizer
tfidf_vect = TfidfVectorizer(max_features=5000)

# Vetorizando
X_tfidf = tfidf_vect.fit_transform(texto)

In [ ]:
print(X_tfidf)

# Modelos

Primeiro, é preciso dividir os dados em base de treino (70%) e teste (30%).

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X1_train, X1_test, y1_train, y1_test = train_test_split(X_bow, data['label'],
                                                        test_size=0.3, random_state = 10)

X2_train, X2_test, y2_train, y2_test = train_test_split(X_tfidf, data['label'],
                                                        test_size=0.3, random_state = 10)

Importando as métricas que serão usadas para avaliação de cada modelo:

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score

In [ ]:
def mostra_metricas(y_true, y_pred):
  ''' Função que recebe o y real, o y predito e mostra as
  principais metricas.
  '''
  print("Acurácia: ", accuracy_score(y_true, y_pred))
  print("\nAUROC:", roc_auc_score(y_true, y_pred))
  print("\nF1-Score:", f1_score(y_true, y_pred, average='weighted'))
  print("\nMatriz de confusão:")
  sns.heatmap(confusion_matrix(y_true, y_pred), annot=True)
  plt.show()

## Regressão Logística

### Texto vetorizado com Bag of Words

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
# Instanciando a reg. logistica
reglog = LogisticRegression()

# Aplicando o modelo
reglog.fit(X1_train, y1_train)

In [ ]:
# Predicao
y1_reglog_pred = reglog.predict(X1_test)

Vamos agora analisar as métricas:

In [ ]:
mostra_metricas(y1_test, y1_reglog_pred)

### Texto vetorizado com tf-idf

In [ ]:
reglog2 = LogisticRegression()

reglog2.fit(X2_train, y2_train)

y2_reglog_pred = reglog2.predict(X2_test)

In [ ]:
mostra_metricas(y2_test, y2_reglog_pred)

A diferença do desempenho do modelo com os 2 métodos de feature extraction é pouca, mas todas as métricas apontam ele foi melhor com tf-idf.

## Naive Bayes Multinomial

### BoW

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
nb1 = MultinomialNB()

nb1.fit(X1_train.toarray(), y1_train)

y1_gnb_pred = nb1.predict(X1_test.toarray())

mostra_metricas(y1_test, y1_gnb_pred)

### Tf-idf

In [ ]:
nb2 = MultinomialNB()

nb2.fit(X2_train.toarray(), y2_train)

y2_gnb_pred = nb2.predict(X2_test.toarray())

mostra_metricas(y2_test, y2_gnb_pred)

## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

### BoW

In [ ]:
rf1 = RandomForestClassifier()

rf1.fit(X1_train, y1_train)

y1_dt_pred = rf1.predict(X1_test)

mostra_metricas(y1_test, y1_dt_pred)

### Tf-idf

In [ ]:
rf2 = RandomForestClassifier()

rf2.fit(X2_train, y2_train)

y2_dt_pred = rf2.predict(X2_test)

mostra_metricas(y2_test, y2_dt_pred)

# Modelos clássicos

### 1. Preparação dos dados para Deep Learning


### Modelo de Deep Learning - LSTM

In [ ]:
X = data['review']
y = data['label']

# Garantir que não tem nulos
X = X.fillna('')

# 2. Dividir treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

### Tokenização

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 10000

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

### 4. Padding

In [ ]:
max_len = 100

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

### 5. Modelo LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

model = Sequential()

model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model.add(LSTM(64))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))

model.summary()

### 6. Compilação e Treinemanto

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

### 8. Avaliação

In [ ]:
loss, accuracy = model.evaluate(X_test_pad, y_test)
print(f"Acurácia no teste (LSTM): {accuracy:.4f}")

### 9. Previsões

In [ ]:
y_pred_prob = model.predict(X_test_pad)
y_pred = (y_pred_prob > 0.5).astype(int)

### Acurácia

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Acurácia do modelo LSTM')
plt.ylabel('Acurácia')
plt.xlabel('Épocas')
plt.legend(['Treino', 'Validação'])
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Loss do modelo LSTM')
plt.ylabel('Loss')
plt.xlabel('Épocas')
plt.legend(['Treino', 'Validação'])
plt.show()

In [ ]:
from sklearn.metrics import classification_report

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d')
plt.title('Matriz de Confusão - LSTM')
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.show()

print(classification_report(y_test, y_pred))

# Comparação entre abordagens

Os modelos tradicionais baseados em TF-IDF apresentaram bom desempenho, com a vantagem de menor custo computacional e maior simplicidade de implementação. Esses modelos se mostraram eficientes para o problema proposto, mesmo utilizando representações mais simples do texto.

Por outro lado, o modelo de Deep Learning baseado em LSTM demonstrou a capacidade de capturar a ordem e o contexto das palavras, característica importante em tarefas de processamento de linguagem natural. No entanto, esse modelo apresentou maior custo computacional e tempo de treinamento, além de exigir maior volume de dados para atingir seu potencial máximo.

## Resultados

De forma geral, a diferença entre as abordagens Bag of Words e TF-IDF foi pequena. Ainda assim, o TF-IDF apresentou desempenho ligeiramente superior na maioria dos modelos, com exceção do Naive Bayes, que teve melhor desempenho com Bag of Words.

Entre os modelos avaliados, a Regressão Logística com TF-IDF apresentou o melhor desempenho geral, atingindo aproximadamente 90% de acurácia e F1-score, além de AUROC de 89%. Esse resultado indica que, para o conjunto de dados utilizado, abordagens mais simples já são suficientes para obter alto desempenho.

O modelo LSTM, apesar de mais sofisticado, não apresentou ganho significativo em relação aos modelos tradicionais, o que pode estar relacionado ao tamanho do dataset e à necessidade de maior ajuste de hiperparâmetros.

A seguir, é apresentada uma função para testar novas previsões utilizando o melhor modelo encontrado.

In [ ]:
def nova_predicao(texto):
  '''Funcao que recebe uma string e printa a predicao feita
  pelo modelo reglog2.'''

    #O modelo 'reglog2' analisa os pesos das palavras no vetor e calcula a 
    # probabilidade de pertencer à classe 0 (Negativo) ou 1 (Positivo).
    
  texto_vetorizado = tfidf_vect.transform([texto])
  pred = reglog2.predict(texto_vetorizado)

    # Tradução do resultado numérico para linguagem humana
    # Como o predict retorna um array (ex: [0]), usamos pred[0] para pegar o valor.
  if pred[0] == 0:
    print("Essa é uma review negativa.")
  else:
    print("Essa é uma review positiva.")


In [ ]:
nova_predicao("Demorou muito não gostei")

In [ ]:
nova_predicao("Achei cheirosinho")

In [ ]:
nova_predicao("Gostei")

In [ ]:
nova_predicao("Não gostei")

In [ ]:
nova_predicao("Nossa que produto ruim é esse parece que encontrei no lixo")

In [ ]:
nova_predicao("Gostei mas não gostei")

In [ ]:
def nova_predicao_lstm(texto):
    '''Função que recebe uma string e realiza a predição 
    utilizando o modelo LSTM treinado acima.'''
    
    # 2. Transformar o texto em sequência de números usando o SEU tokenizer
    sequencia = tokenizer.texts_to_sequences([texto])
    
    # 3. Aplicar o Padding (preenchimento) para ter o tamanho 100
    sequencia_pad = pad_sequences(sequencia, maxlen=100)
    
    # 4. Fazer a predição
    pred_prob = model.predict(sequencia_pad, verbose=0)
    
    # Como a última camada é 'sigmoid', o resultado é uma probabilidade entre 0 e 1
    if pred_prob[0][0] > 0.5:
        print(f"Review POSITIVA (Confiança: {pred_prob[0][0]:.2f})")
    else:
        print(f"Review NEGATIVA (Confiança: {1 - pred_prob[0][0]:.2f})")

In [ ]:
nova_predicao_lstm("O produto é maravilhoso, chegou muito rápido!")

In [ ]:
nova_predicao_lstm("Péssima experiência, o produto veio quebrado.")

In [ ]:
nova_predicao_lstm("O produto chegou antes do prazo e em perfeitas condições!")

In [ ]:
nova_predicao_lstm("Gostei mas não gostei")

Apesar do bom desempenho dos modelos tradicionais, o uso de Deep Learning demonstra maior potencial em cenários com maior volume de dados e maior complexidade semântica.

## Referências

HOCHREITER, S.; SCHMIDHUBER, J. Long Short-Term Memory. Neural Computation, v. 9, n. 8, p. 1735–1780, 1997.

MIKOLOV, T. et al. Efficient Estimation of Word Representations in Vector Space. arXiv preprint arXiv:1301.3781, 2013.

JURAFSKY, D.; MARTIN, J. H. Speech and Language Processing. 3rd ed. Draft, 2023.
